# Sigma Parameter Interrogation

Diagnostic notebook for understanding how the Langevin diffusion noise
parameter `sigma` shapes the d' curve across all ISIs.

**Fixed parameters:**
- `sigma0 = 0.193774` (encoding noise, from Stage A fit)
- `drift_step_size = 0.01` (prior drift)

**Analyses:**
1. Sweep sigma across a wide range, evaluate d' at every ISI [1..64]
2. Dose-response: d' vs sigma for each ISI separately
3. MSE vs sigma: where is the best-fit value?
4. 2D interaction: sigma x drift_step_size heatmap

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
from functools import partial

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.params import model_params as model_params_tm
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import (
    run_experiment_scores_prior,
    make_noise_schedule,
)
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
    make_high_diversity_sequences,
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    plot_sigma_fit,
    _compute_auroc_upper_envelope,
    auc_to_dprime as auc2dp,
)

from ScoreFunction import ScoreFunction

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

## 1. Load config & human data

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


CONFIG_PATH = (
    "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)
model_cfg, model_cfg_path = load_config(CONFIG_PATH)

exp_cfg = model_cfg["experiment"]
which_task = 2
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)
isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]
metric = model_cfg["metric"]
noise_mode = "constant"

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(2, 0, True, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"Stimulus pool size: {len(stimulus_pool)}")

# Human targets by ISI (for MSE computation)
isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}
sigma_human = {
    isi: float(human_curve[isi_to_hc_idx[isi]])
    for isi in [1, 2, 4, 8, 16, 32, 64]
}
print("\nHuman d' targets:")
for isi, dp in sigma_human.items():
    print(f"  ISI {isi:>2}: {dp:.4f}")

## 2. Build encoder & encode stimuli

In [ ]:
pc_dims = 256
device = 'cuda'

pc_texture_model = AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.01,
    duration=2.0,
    device=device
)

X = encode_stimuli(pc_texture_model, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

## 3. Load score function (prior)

In [ ]:
SCORE_CONFIG = "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/bryan.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

score_fn = ScoreFunction(
    mode="textures",
    restart=True,
    config=SCORE_CONFIG,
    device=DEVICE,
)

test_input = torch.randn(1, 256, device=DEVICE)
test_output = score_fn.forward(test_input)
print(f"Score output shape: {test_output.shape}")
print(f"Score output norm: {test_output.view(-1).norm():.4f} (should be ~1.0)")

## 4. Fixed parameters

In [ ]:
SIGMA0 = 0.193774
DRIFT_STEP_SIZE = 0.01

print(f"Fixed sigma0 (encoding noise): {SIGMA0}")
print(f"Fixed drift_step_size:         {DRIFT_STEP_SIZE}")

## 5. Generate multi-ISI sequences

Using `make_high_diversity_sequences` across all ISIs [1, 2, 4, 8, 16, 32, 64]
so that sigma's effect is visible at long delays where many Langevin steps accumulate.

In [ ]:
ISI_VALUES = [1, 2, 4, 8, 16, 32, 64]

sigma_exps, sigma_isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=ISI_VALUES,
    n_sequences=10,
    length=150,
    min_pairs_per_isi=5,
    seed=1000,
)

print(f"Generated {len(sigma_exps)} sequences")
print(f"Trials per sequence: {len(sigma_exps[0])}")

# Precompute ISIs for each sequence
seq_trial_isis = [infer_trial_isis(seq) for seq in sigma_exps]

# Validate ISI distribution
isi_counts = defaultdict(list)
for seq in sigma_exps:
    counts = Counter(infer_trial_isis(seq))
    for isi_val in ISI_VALUES:
        isi_counts[isi_val].append(counts.get(isi_val, 0))

print("\nPairs per ISI per sequence (mean +/- std):")
for isi_val in ISI_VALUES:
    vals = isi_counts[isi_val]
    print(f"  ISI {isi_val:>2}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}  "
          f"(min={min(vals)}, max={max(vals)})")

## 6. Evaluation helper

In [ ]:
def evaluate_sigma_drift(
    sigma_val, drift_val,
    n_mc=8, n_seqs_per_rep=5,
    isi_values=ISI_VALUES,
):
    """Evaluate a (sigma, drift_step_size) pair across all ISIs.

    Returns
    -------
    mean_mse : float
        Average MSE vs human d' across ISIs and MC reps.
    mean_dprimes : dict[int, float]
        Mean model d' per ISI.
    """
    run_fn = partial(
        run_experiment_scores_prior,
        score_model=score_fn,
        drift_step_size=drift_val,
        sigma=sigma_val,
    )
    mse_reps = []
    dprime_reps = {isi: [] for isi in isi_values}

    for rep in range(n_mc):
        rng = np.random.default_rng(300_000 + rep)
        seq_idx = rng.choice(
            len(sigma_exps),
            size=min(n_seqs_per_rep, len(sigma_exps)),
            replace=False,
        )

        all_hits, all_isis_for_hits, all_fas = [], [], []
        for si in seq_idx:
            run_out = run_fn(
                sigma0=SIGMA0,
                noise_mode=noise_mode,
                metric=metric,
                X0=X,
                name_to_idx=name_to_idx,
                experiment_list=[sigma_exps[si]],
                debug=False,
                seed=300_000 + rep * 1000 + int(si),
            )
            h = np.asarray(run_out["hits"])
            f = np.asarray(run_out["fas"])
            t_isis = seq_trial_isis[si]
            if len(h) != len(t_isis):
                continue
            all_hits.append(h)
            all_isis_for_hits.extend(t_isis)
            all_fas.append(f)

        if not all_hits:
            continue
        hits_arr = np.concatenate(all_hits)
        isis_arr = np.array(all_isis_for_hits)
        fas_arr = np.concatenate(all_fas) if all_fas else np.array([])
        if len(fas_arr) == 0:
            continue

        rep_mse = []
        for isi_val in isi_values:
            mask = isis_arr == isi_val
            hits_isi = hits_arr[mask]
            if len(hits_isi) == 0:
                continue
            human_dp = sigma_human.get(isi_val)
            if human_dp is None:
                continue
            auroc = _compute_auroc_upper_envelope(hits_isi, fas_arr)
            dp = auc2dp(auroc, eps=1e-4)
            dprime_reps[isi_val].append(dp)
            rep_mse.append((dp - human_dp) ** 2)

        if rep_mse:
            mse_reps.append(np.mean(rep_mse))

    mean_mse = np.mean(mse_reps) if mse_reps else np.inf
    mean_dprimes = {
        isi: np.mean(v) if v else np.nan
        for isi, v in dprime_reps.items()
    }
    return mean_mse, mean_dprimes


print("Evaluation helper defined.")

---
## 7. Sweep sigma (fixed drift = 0.01)

Sweep sigma across a wide log-spaced range and evaluate d' at every ISI.
At low ISIs (1-2 steps), sigma should have little effect.
At high ISIs (32-64 steps), sigma's stochastic noise accumulates and should
visibly degrade d'.

In [ ]:
SIGMA_GRID = np.geomspace(0.001, 5.0, 12)
N_MC = 8
N_SEQS_PER_REP = 5

print(f"Sweeping {len(SIGMA_GRID)} sigma values: {SIGMA_GRID.round(4)}")
print(f"Fixed drift_step_size = {DRIFT_STEP_SIZE}")
print(f"MC reps per sigma = {N_MC}, seqs per rep = {N_SEQS_PER_REP}")

sigma_sweep_results = []

with torch.no_grad():
    for sigma_val in tqdm(SIGMA_GRID, desc="sigma sweep"):
        mse, dprimes = evaluate_sigma_drift(
            sigma_val, DRIFT_STEP_SIZE,
            n_mc=N_MC, n_seqs_per_rep=N_SEQS_PER_REP,
        )
        sigma_sweep_results.append({
            "sigma": sigma_val,
            "mse": mse,
            "dprimes": dprimes,
        })
        print(f"  sigma={sigma_val:.4f}  MSE={mse:.4f}  "
              f"d'@ISI1={dprimes.get(1, np.nan):.3f}  "
              f"d'@ISI64={dprimes.get(64, np.nan):.3f}")
        torch.cuda.empty_cache()

print("\nSweep complete.")

## 8. Plot 1: d' vs ISI curves for each sigma

Each line is a different sigma value. Human data in black.
Key question: does increasing sigma steepen the d' decay at long ISIs?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

cmap = plt.cm.viridis
norm = plt.Normalize(
    vmin=np.log10(SIGMA_GRID.min()),
    vmax=np.log10(SIGMA_GRID.max()),
)

x_pos = np.arange(len(ISI_VALUES))

for res in sigma_sweep_results:
    dp_vals = [res["dprimes"].get(isi, np.nan) for isi in ISI_VALUES]
    color = cmap(norm(np.log10(res["sigma"])))
    ax.plot(x_pos, dp_vals, 'o-', color=color, alpha=0.7,
            label=f'sigma={res["sigma"]:.4f}')

# Human reference
human_dp_subset = [sigma_human[isi] for isi in ISI_VALUES]
ax.plot(x_pos, human_dp_subset, 'k*-', markersize=12, linewidth=2.5,
        label='Human', zorder=10)

ax.set_xticks(x_pos)
ax.set_xticklabels([str(i) for i in ISI_VALUES])
ax.set_xlabel('ISI (intervening items)')
ax.set_ylabel("d'")
ax.set_title(f"Effect of sigma on d' curve (sigma0={SIGMA0}, drift={DRIFT_STEP_SIZE})")
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Plot 2: d' vs sigma for each ISI (dose-response)

Each line is a different ISI. Human targets shown as horizontal dashed lines.
Shows how sensitive each ISI is to changes in sigma.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.coolwarm(np.linspace(0, 1, len(ISI_VALUES)))
sigma_vals = [r["sigma"] for r in sigma_sweep_results]

for i, isi_val in enumerate(ISI_VALUES):
    dp_vals = [r["dprimes"].get(isi_val, np.nan) for r in sigma_sweep_results]
    ax.plot(sigma_vals, dp_vals, 'o-', color=colors[i], label=f'ISI {isi_val}')
    # Human target
    ax.axhline(sigma_human[isi_val], color=colors[i], ls='--', alpha=0.4)

ax.set_xscale('log')
ax.set_xlabel('sigma (Langevin noise)')
ax.set_ylabel("d'")
ax.set_title(f"Dose-response: d' vs sigma per ISI (sigma0={SIGMA0}, drift={DRIFT_STEP_SIZE})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Plot 3: MSE vs sigma

Overall MSE(sigma) aggregated across all ISIs.
If sigma matters, there should be a clear minimum.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sigma_vals = [r["sigma"] for r in sigma_sweep_results]
mse_vals = [r["mse"] for r in sigma_sweep_results]

ax.plot(sigma_vals, mse_vals, 'o-', color='tab:blue', linewidth=2, markersize=8)

best_idx = np.argmin(mse_vals)
ax.axvline(sigma_vals[best_idx], color='red', ls='--', alpha=0.7,
           label=f'best sigma={sigma_vals[best_idx]:.4f} (MSE={mse_vals[best_idx]:.4f})')

ax.set_xscale('log')
ax.set_xlabel('sigma (Langevin noise)')
ax.set_ylabel('MSE vs human d\'')
ax.set_title(f'MSE vs sigma (sigma0={SIGMA0}, drift={DRIFT_STEP_SIZE})')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

---
## 11. 2D interaction: sigma x drift_step_size

Sweep a grid of (sigma, drift_step_size) pairs to see whether these
parameters trade off or act independently.

In [ ]:
SIGMA_GRID_2D = np.geomspace(0.01, 3.0, 6)
DRIFT_GRID_2D = np.geomspace(0.005, 0.5, 6)
N_MC_2D = 6
N_SEQS_2D = 5

print(f"2D sweep: {len(SIGMA_GRID_2D)} sigma x {len(DRIFT_GRID_2D)} drift = "
      f"{len(SIGMA_GRID_2D) * len(DRIFT_GRID_2D)} evaluations")
print(f"Sigma grid:  {SIGMA_GRID_2D.round(4)}")
print(f"Drift grid:  {DRIFT_GRID_2D.round(4)}")

mse_grid_2d = np.full((len(DRIFT_GRID_2D), len(SIGMA_GRID_2D)), np.nan)
interaction_results = []

with torch.no_grad():
    total = len(SIGMA_GRID_2D) * len(DRIFT_GRID_2D)
    pbar = tqdm(total=total, desc="2D sweep")
    for j, sigma_val in enumerate(SIGMA_GRID_2D):
        for i, drift_val in enumerate(DRIFT_GRID_2D):
            mse, dprimes = evaluate_sigma_drift(
                sigma_val, drift_val,
                n_mc=N_MC_2D, n_seqs_per_rep=N_SEQS_2D,
            )
            mse_grid_2d[i, j] = mse
            interaction_results.append({
                "sigma": sigma_val,
                "drift": drift_val,
                "mse": mse,
                "dprimes": dprimes,
            })
            pbar.update(1)
            torch.cuda.empty_cache()
    pbar.close()

# Find best
best_2d_idx = np.unravel_index(np.nanargmin(mse_grid_2d), mse_grid_2d.shape)
print(f"\nBest: sigma={SIGMA_GRID_2D[best_2d_idx[1]]:.4f}, "
      f"drift={DRIFT_GRID_2D[best_2d_idx[0]]:.4f}, "
      f"MSE={mse_grid_2d[best_2d_idx]:.4f}")

## 12. Plot 4: MSE heatmap (sigma x drift)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    mse_grid_2d,
    origin='lower',
    aspect='auto',
    cmap='viridis_r',
    extent=[
        np.log10(SIGMA_GRID_2D[0]), np.log10(SIGMA_GRID_2D[-1]),
        np.log10(DRIFT_GRID_2D[0]), np.log10(DRIFT_GRID_2D[-1]),
    ],
)
cbar = fig.colorbar(im, ax=ax, label='MSE vs human d\'')

# Mark best point
ax.plot(
    np.log10(SIGMA_GRID_2D[best_2d_idx[1]]),
    np.log10(DRIFT_GRID_2D[best_2d_idx[0]]),
    'r*', markersize=15, label='best'
)

ax.set_xlabel('log10(sigma)')
ax.set_ylabel('log10(drift_step_size)')
ax.set_title(f'MSE(sigma, drift) heatmap (sigma0={SIGMA0})')
ax.legend()
fig.tight_layout()
plt.show()

# Also show as text table
print("\nMSE grid (rows=drift, cols=sigma):")
print(f"{'':>10}", end="")
for s in SIGMA_GRID_2D:
    print(f"  {s:>8.4f}", end="")
print()
for i, d in enumerate(DRIFT_GRID_2D):
    print(f"d={d:<8.4f}", end="")
    for j in range(len(SIGMA_GRID_2D)):
        print(f"  {mse_grid_2d[i, j]:>8.4f}", end="")
    print()

## 13. Summary

Key findings from this analysis.

In [ ]:
# Best sigma from 1D sweep
best_1d_idx = np.argmin([r["mse"] for r in sigma_sweep_results])
best_1d = sigma_sweep_results[best_1d_idx]

print("=" * 60)
print("SIGMA PARAMETER INTERROGATION SUMMARY")
print("=" * 60)
print(f"\nFixed parameters:")
print(f"  sigma0           = {SIGMA0}")
print(f"  drift_step_size  = {DRIFT_STEP_SIZE}")
print(f"\n1D sweep (drift fixed at {DRIFT_STEP_SIZE}):")
print(f"  Best sigma = {best_1d['sigma']:.6f}  (MSE = {best_1d['mse']:.4f})")
print(f"  d' by ISI: {best_1d['dprimes']}")
print(f"\n2D sweep (best joint fit):")
print(f"  Best sigma = {SIGMA_GRID_2D[best_2d_idx[1]]:.6f}")
print(f"  Best drift = {DRIFT_GRID_2D[best_2d_idx[0]]:.6f}")
print(f"  MSE        = {mse_grid_2d[best_2d_idx]:.4f}")